# Hybrid Ensemble Hyperparameter Tuning

This tuned version treats the ensemble member weights as hyperparameters. It uses the same official CIFAKE split discipline as the tuned single-model notebooks: ensemble-weight selection uses a stratified validation subset from the official training split, and final evaluation is performed only on the official test split.


In [ ]:
from pathlib import Path
from itertools import product
import json
import os
import importlib.util
import random
import subprocess
import sys

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from PIL import Image, ImageFile
from tqdm.auto import tqdm

try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
pin_memory = device.type == "cuda"
num_workers = 0
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA GPU was not detected. The notebook will run on CPU unless you install/use a CUDA-enabled PyTorch environment.")

IN_COLAB = os.getenv("COLAB_RELEASE_TAG") is not None or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)


In [ ]:
# Mount Google Drive in Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive", force_remount=True)


In [ ]:
def has_cifake_splits(data_dir):
    data_dir = Path(data_dir).expanduser().resolve()
    return (data_dir / "train").exists() and (data_dir / "test").exists()


def find_project_root():
    if IN_COLAB:
        return Path("/content/TDK_AI_Detection")

    cwd = Path.cwd().resolve()
    candidates = []

    env_project_root = os.getenv("TDK_PROJECT_ROOT")
    if env_project_root:
        candidates.append(Path(env_project_root).expanduser())

    env_data_root = os.getenv("CIFAKE_ROOT")
    if env_data_root:
        data_root = Path(env_data_root).expanduser()
        if has_cifake_splits(data_root):
            return data_root.resolve().parent
        candidates.append(data_root)

    candidates.extend([cwd, *list(cwd.parents)[:4]])
    candidates.extend([
        Path("/content/TDK_AI_Detection"),
        Path("/content/gdrive/MyDrive/TDK_AI_Detection"),
        Path("/content/drive/MyDrive/TDK_AI_Detection"),
    ])

    seen = set()
    for candidate in candidates:
        candidate = Path(candidate).expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_cifake_splits(candidate / "CIFAKE_FULL"):
            return candidate

    if cwd.name.lower() == "tuned models":
        return cwd.parent
    return cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "CIFAKE_FULL"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
MODEL_DIR = PROJECT_ROOT / "models" / "tuned"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tuned"
HYBRID_DIR = OUTPUT_DIR / "hybrid"
PROBABILITY_CACHE_DIR = HYBRID_DIR / "probability_cache"
HYBRID_DIR.mkdir(parents=True, exist_ok=True)
PROBABILITY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError("Expected CIFAKE_FULL/train and CIFAKE_FULL/test under the project root.")

class_names = ["fake", "real"]
normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]


def make_transforms(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=normalize_mean, std=normalize_std),
    ])


def is_valid_image_file(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def normalized_imagefolder(split_dir, image_size):
    raw_unfiltered = datasets.ImageFolder(split_dir)
    raw = datasets.ImageFolder(
        split_dir,
        transform=make_transforms(image_size),
        is_valid_file=is_valid_image_file,
    )
    skipped = len(raw_unfiltered.samples) - len(raw.samples)
    if skipped:
        print(f"Skipped {skipped} unreadable image(s) in {split_dir}.")
    lower_to_original = {name.lower(): idx for name, idx in raw.class_to_idx.items()}
    fake_idx = lower_to_original["fake"]
    real_idx = lower_to_original["real"]
    remap = {fake_idx: 0, real_idx: 1}
    raw.target_transform = lambda target: remap[target]
    normalized_targets = np.array([remap[target] for _, target in raw.samples])
    return raw, normalized_targets


def make_tuning_splits(image_size, train_per_class=2000, val_per_class=500):
    dataset, targets = normalized_imagefolder(TRAIN_DIR, image_size)
    rng = np.random.default_rng(seed)
    train_idx = []
    val_idx = []
    for label in [0, 1]:
        label_idx = np.where(targets == label)[0]
        chosen = rng.choice(label_idx, size=train_per_class + val_per_class, replace=False)
        train_idx.extend(chosen[:train_per_class].tolist())
        val_idx.extend(chosen[train_per_class:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def metrics_from_predictions(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_fake_positive": float(precision_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "recall_fake_positive": float(recall_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "f1_fake_positive": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "precision_real_positive": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_real_positive": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_real_positive": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "weighted_precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


In [ ]:
MODEL_NAME = "hybrid_ensemble_hyperparameter_tuned"
TUNING_TRAIN_PER_CLASS = 2000
TUNING_VAL_PER_CLASS = 500
BASE_IMAGE_SIZE_FOR_SPLIT = 224
CACHE_FORMAT_VERSION = 2

candidate_values = np.round(np.linspace(0.0, 1.0, 21), 2).tolist()
TUNING_RESULTS_PATH = HYBRID_DIR / "hybrid_weight_tuning_results.json"
BEST_CONFIG_PATH = HYBRID_DIR / "hybrid_hyperparameter_tuned_best_config.json"
FINAL_RESULTS_PATH = HYBRID_DIR / "hybrid_hyperparameter_tuned_final_results.json"
BEST_PREDICTIONS_PATH = HYBRID_DIR / "hybrid_hyperparameter_tuned_predictions.npz"
print("Candidate ensemble weight values:", candidate_values)

model_names = ["resnet50", "efficientnetv2s", "vit16", "xception"]
model_artifacts = {
    "resnet50": {
        "config": OUTPUT_DIR / "resnet50_hyperparameter_tuned_best_config.json",
        "checkpoint": MODEL_DIR / "resnet50_hyperparameter_tuned_best.pth",
    },
    "efficientnetv2s": {
        "config": OUTPUT_DIR / "efficientnetv2s_hyperparameter_tuned_best_config.json",
        "checkpoint": MODEL_DIR / "efficientnetv2s_hyperparameter_tuned_best.pth",
    },
    "vit16": {
        "config": OUTPUT_DIR / "vit16_hyperparameter_tuned_best_config.json",
        "checkpoint": MODEL_DIR / "vit16_hyperparameter_tuned_best.pth",
    },
    "xception": {
        "config": OUTPUT_DIR / "xception_hyperparameter_tuned_best_config.json",
        "checkpoint": MODEL_DIR / "xception_hyperparameter_tuned_best.pth",
    },
}


In [ ]:
def build_resnet50_model():
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def build_efficientnetv2s_model():
    model = models.efficientnet_v2_s(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 2)
    return model


def build_vit16_model():
    model = models.vit_b_16(weights=None)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, 2)
    return model


def build_xception_model():
    return timm.create_model("xception", pretrained=False, num_classes=2)


model_builders = {
    "resnet50": build_resnet50_model,
    "efficientnetv2s": build_efficientnetv2s_model,
    "vit16": build_vit16_model,
    "xception": build_xception_model,
}


def load_best_hyperparameters(model_name):
    config_path = model_artifacts[model_name]["config"]
    if not config_path.exists():
        raise FileNotFoundError(f"Missing {config_path}. Run the tuned notebook for {model_name} first.")
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    return config["hyperparameters"]


def load_tuned_model(model_name):
    checkpoint_path = model_artifacts[model_name]["checkpoint"]
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing {checkpoint_path}. Run the tuned notebook for {model_name} first.")
    model = model_builders[model_name]().to(device)
    try:
        state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        state_dict = torch.load(checkpoint_path, map_location=device)
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
    model.load_state_dict(state_dict)
    model.eval()
    return model


@torch.no_grad()
def predict_probabilities(model, loader):
    probs = []
    targets = []
    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        outputs = model(images)
        batch_probs = torch.softmax(outputs, dim=1).cpu().numpy()
        probs.append(batch_probs)
        targets.extend(labels.numpy().tolist())
    return np.concatenate(probs, axis=0), np.array(targets)


In [ ]:
def probability_cache_path(model_name, split_name):
    return PROBABILITY_CACHE_DIR / f"{model_name}_{split_name}_tuned_probs.npz"


def get_probabilities(model_name, split_name, dataset, subset_indices=None, batch_size=16):
    cache_path = probability_cache_path(model_name, split_name)
    checkpoint_stat = model_artifacts[model_name]["checkpoint"].stat()
    expected_count = len(dataset) if subset_indices is None else len(subset_indices)
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as data:
            probs = data["probs"]
            targets = data["targets"]
            cache_valid = (
                int(data.get("cache_format_version", -1)) == CACHE_FORMAT_VERSION
                and int(data.get("checkpoint_size", -1)) == checkpoint_stat.st_size
                and int(data.get("checkpoint_mtime_ns", -1)) == checkpoint_stat.st_mtime_ns
                and probs.shape == (expected_count, 2)
                and targets.shape == (expected_count,)
                and np.isfinite(probs).all()
            )
        if cache_valid:
            print(f"Loaded validated {model_name} {split_name} probabilities from:", cache_path)
            return probs, targets
        print("Ignoring stale or invalid probability cache:", cache_path)

    model = load_tuned_model(model_name)
    loader_dataset = dataset if subset_indices is None else Subset(dataset, subset_indices)
    loader = DataLoader(loader_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    probs, targets = predict_probabilities(model, loader)
    if not np.isfinite(probs).all():
        raise FloatingPointError(f"Non-finite probabilities produced for {model_name} {split_name}")
    np.savez_compressed(
        cache_path,
        probs=probs,
        targets=targets,
        cache_format_version=np.array(CACHE_FORMAT_VERSION, dtype=np.int64),
        checkpoint_size=np.array(checkpoint_stat.st_size, dtype=np.int64),
        checkpoint_mtime_ns=np.array(checkpoint_stat.st_mtime_ns, dtype=np.int64),
    )
    print(f"Saved {model_name} {split_name} probabilities to:", cache_path)
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return probs, targets


def build_probability_stacks():
    _, val_idx = make_tuning_splits(
        BASE_IMAGE_SIZE_FOR_SPLIT,
        train_per_class=TUNING_TRAIN_PER_CLASS,
        val_per_class=TUNING_VAL_PER_CLASS,
    )

    val_probs_by_model = {}
    test_probs_by_model = {}
    y_val = None
    y_test = None

    for model_name in model_names:
        hp = load_best_hyperparameters(model_name)
        image_size = hp.get("image_size", BASE_IMAGE_SIZE_FOR_SPLIT)
        batch_size = hp.get("batch_size", 16)
        val_dataset, _ = normalized_imagefolder(TRAIN_DIR, image_size)
        test_dataset, _ = normalized_imagefolder(TEST_DIR, image_size)

        val_probs, current_y_val = get_probabilities(model_name, "val", val_dataset, subset_indices=val_idx, batch_size=batch_size)
        test_probs, current_y_test = get_probabilities(model_name, "test", test_dataset, subset_indices=None, batch_size=batch_size)

        if y_val is None:
            y_val = current_y_val
        elif not np.array_equal(y_val, current_y_val):
            raise ValueError(f"Validation target order mismatch for {model_name}")

        if y_test is None:
            y_test = current_y_test
        elif not np.array_equal(y_test, current_y_test):
            raise ValueError(f"Test target order mismatch for {model_name}")

        val_probs_by_model[model_name] = val_probs
        test_probs_by_model[model_name] = test_probs

    val_stack = np.stack([val_probs_by_model[name] for name in model_names], axis=0)
    test_stack = np.stack([test_probs_by_model[name] for name in model_names], axis=0)
    return val_stack, y_val, test_stack, y_test


val_stack, y_val, test_stack, y_test = build_probability_stacks()
print("Validation probability stack:", val_stack.shape)
print("Test probability stack:", test_stack.shape)


In [ ]:
results = []
best = None
for raw_weights in product(candidate_values, repeat=len(model_names)):
    total = sum(raw_weights)
    if total <= 0:
        continue
    weights = np.array(raw_weights, dtype=np.float64) / total
    blended = np.tensordot(weights, val_stack, axes=(0, 0))
    pred = blended.argmax(axis=1)
    metrics = metrics_from_predictions(y_val, pred)
    row = {
        "raw_weights": dict(zip(model_names, [float(x) for x in raw_weights])),
        "normalized_weights": dict(zip(model_names, [float(x) for x in weights])),
        "validation_metrics": metrics,
    }
    results.append(row)
    if best is None or metrics["weighted_f1"] > best["validation_metrics"]["weighted_f1"]:
        best = row

with open(TUNING_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
with open(BEST_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(best, f, indent=4)

print("Best tuned ensemble weights:")
print(json.dumps(best, indent=2))


In [ ]:
best_weights = np.array([best["normalized_weights"][name] for name in model_names], dtype=np.float64)
test_probs = np.tensordot(best_weights, test_stack, axes=(0, 0))
y_pred = test_probs.argmax(axis=1)
test_metrics = metrics_from_predictions(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
report_dict = classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_text = classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=class_names,
    zero_division=0,
)

final_results = {
    "model": MODEL_NAME,
    "tuned_hyperparameter": "ensemble_member_weights",
    "tuning_design": {
        "tuning_train_images": TUNING_TRAIN_PER_CLASS * 2,
        "tuning_validation_images": TUNING_VAL_PER_CLASS * 2,
        "final_test_split": str(TEST_DIR),
    },
    "candidate_values": candidate_values,
    "members": model_names,
    "best_validation": best,
    "official_test_metrics": test_metrics,
    "confusion_matrix": cm.tolist(),
    "classification_report": report_dict,
}
with open(FINAL_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=4)
np.savez_compressed(
    BEST_PREDICTIONS_PATH,
    y_true=y_test,
    y_pred=y_pred,
    hybrid_probs=test_probs,
    weights=best_weights,
    model_names=np.array(model_names),
)

print("Official test metrics:")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Precision, fake positive: {test_metrics['precision_fake_positive']:.4f}")
print(f"Recall, fake positive: {test_metrics['recall_fake_positive']:.4f}")
print(f"F1, fake positive: {test_metrics['f1_fake_positive']:.4f}")
print(f"Weighted precision: {test_metrics['weighted_precision']:.4f}")
print(f"Weighted recall: {test_metrics['weighted_recall']:.4f}")
print(f"Weighted F1: {test_metrics['weighted_f1']:.4f}")
print("Confusion matrix [[fake->fake, fake->real], [real->fake, real->real]]:")
print(cm)
print("Classification report:")
print(report_text)
print("Saved final tuned hybrid results to:", FINAL_RESULTS_PATH)
